# Batch API synthesis: which kinetic constants shape the product and the impurity?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zuhairblr/middoe/blob/main/tests/tutorials/02_sensitivity/a_kinetic_parameters/tutorial.ipynb)

**Global sensitivity analysis of the parameters** (`sensa`, Sobol indices).

Before estimating anything, ask which of the six Arrhenius parameters actually move the measured
outputs. A parameter the outputs do not feel cannot be estimated from them, whatever the experiment.

You will:

1. compute first-order (`S1`) and total (`ST`) Sobol indices over the full prior bounds;
2. repeat at the other end of the temperature range and see the ranking change;
3. zoom into a ±20 % box around the current guess (`par_d`);
4. follow the indices along the batch and fix the parameters that never matter.

In [ ]:
# Google Colab only: install MIDDoE and fetch the model file(s); does nothing elsewhere
import sys
if 'google.colab' in sys.modules:
    import subprocess, urllib.request
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'git+https://github.com/zuhairblr/middoe.git'], check=True)
    for f in ['model.py']:
        urllib.request.urlretrieve('https://raw.githubusercontent.com/zuhairblr/middoe/main/tests/tutorials/02_sensitivity/a_kinetic_parameters/' + f, f)

### The process

Batch synthesis of a quaternary ammonium salt (an intermediate of an HIV drug):

$$\mathrm{SM + TMA \rightleftharpoons QS1Cl \qquad QS1Cl \rightarrow ClDMI + MeCl}$$

The product QS1Cl forms reversibly and slowly degrades into the impurity ClDMI. Rate and
equilibrium constants follow Arrhenius / van 't Hoff laws around 296.15 K, so the model has six
parameters: three constants at the reference temperature and three energies. Measured: starting
material `y1`, product `y2`, impurity `y3`. Inputs: the temperature profile `u1` and the initial
concentrations `y10`, `y20`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from middoe.sc_sensa import sensa
from middoe.log_utils import save_to_jac

QUICK = False   # True: tiny budgets and fewer designs, a check that everything runs (minutes), not a result.

In [ ]:
theta_true  = [50000, 75000, 0.4116, 111900, 9905, 30000]   # the in-silico 'real' process
theta_guess = [100000, 100000, 1, 100000, 100, 10000]       # starting values
theta_min   = [10000, 0, 0.1, 50000, 10, 10000]             # bounds
theta_max   = [1000000, 200000, 10, 200000, 10000, 200000]
NAMES = ['kf_ref', 'Ef', 'kfs_ref', 'Efs', 'K_ref', 'dH']   # theta1 ... theta6 of model.py

sample = {'init': 0, 'meas': True,  # every output: measured, 17 synchronised samples at least 0.3 h apart,
          'sp': 17, 'offt': 0.3,    # always at 0 and 16 h; noise sd 0.005 mol/L
          'samp_s': 1, 'samp_f': [0, 16], 'unc': 0.005}

system = {
    'tvi': {'u1': {                       # reactor temperature [K]
        'stps': 6, 'cvp': 'LPF', 'const': 'inc',   # 6 levels joined by ramps, rising only
        'min': 296.15, 'max': 306.15,
        'offl': 0.01, 'offt': 0.3,                 # min level change [K], min time between switches [h]
    }},
    'tvo': {'y1': dict(sample), 'y2': dict(sample), 'y3': dict(sample)},  # SM, product, impurity [mol/L]
    'tii': {'y10': {'min': 0.3, 'max': 1},         # initial starting material [mol/L]
            'y20': {'min': 0.19, 'max': 1}},       # initial trimethylamine [mol/L]
    'tio': {},
    't_s': [0, 16],      # 16-h batch
    't_r': 0.1 if QUICK else 0.02,   # time grid [h] (coarser in a quick check)
    't_d': 0.3,          # dead time at both ends [h] (designs only)
}

models = {
    'can_m': ['M'],
    'krt': {'M': 'pys'},                         # a Python file with solve_model(t, y0, tii, tvi, theta)
    'src': {'M': str(Path('model.py').resolve())},
    'theta': {'M': list(theta_guess)},
    't_l': {'M': theta_min},
    't_u': {'M': theta_max},
    'mutation': {'M': [True] * 6},               # True = estimated, False = held at its value
    'theta_names': {'M': NAMES},
}

### The settings of `sensa`

| key | meaning |
|---|---|
| `par_s`, `var_s` | sensitivity to the parameters, to the inputs (here parameters only) |
| `par_d` | `False`: sample each parameter between `t_l` and `t_u`; a float `d`: between `theta/d` and `theta*d` |
| `samp` | Sobol base sample *N*; the model runs *N*(2*D*+2) times for *D* parameters |
| `tii_n`, `tvi_n` | the fixed operating point: `[y10, y20]` and `[u1]` |
| `multi` | share of CPU cores to use (`False`: one core) |

`S1` is the share of output variance a parameter explains alone, `ST` includes its interactions.
`ST` close to 0 means the parameter does not matter at this operating point.

In [ ]:
def gsa_settings(T, **extra):
    gsa = {'par_s': True, 'var_s': False,
           'par_d': False, 'var_d': False,          # sample over models['t_l'] .. models['t_u']
           'samp': 2 ** 5 if QUICK else 2 ** 11,     # Sobol base sample N
           'multi': False if QUICK else 0.7,
           'tii_n': [0.366, 0.19], 'tvi_n': [T],     # operating point: y10, y20 [mol/L]; temperature [K]
           'plt': True}
    gsa.update(extra)
    return gsa


def final_indices(res, model='M'):
    """S1 and ST of every parameter at the end of the batch, one column pair per output."""
    names = NAMES                      # sensa calls them Param_1 ... Param_6, in this order
    cols = {}
    for y, steps in res['analysis'][model].items():
        cols[(y, 'S1')] = steps[-1]['S1']
        cols[(y, 'ST')] = steps[-1]['ST']
    return pd.DataFrame(cols, index=names).round(3)


cold = sensa(gsa_settings(296.15), models, system)
final_indices(cold)

At 296 K the impurity `y3` is governed by the side reaction (`kfs_ref`, `Efs`) and the
equilibrium; the activation energies matter little because the batch stays at the reference
temperature, where the Arrhenius factor is 1. Repeat at 306 K:

In [ ]:
hot = sensa(gsa_settings(306.15), models, system)
pd.concat({'296 K': final_indices(cold).xs('ST', axis=1, level=1),
           '306 K': final_indices(hot).xs('ST', axis=1, level=1)}, axis=1)

Away from the reference temperature the energies `Ef`, `Efs`, `dH` start to matter:
**an experiment at a single temperature cannot identify them.** That is the first hint for the
design (tutorials 4): vary the temperature.

### A local view: `par_d`

The full bounds span two orders of magnitude. With `par_d = 1.2` each parameter is sampled between
`theta/1.2` and `theta*1.2` around `models['theta']`, which is closer to what a fit already knows.

In [ ]:
local = sensa(gsa_settings(306.15, par_d=1.2), models, system)
final_indices(local).xs('ST', axis=1, level=1)

### Along the batch

`sensa` returns the indices at every time of the grid, so you can see *when* a parameter
matters: that is where samples carry information about it.

In [ ]:
steps = hot['analysis']['M']['y3']
t = np.linspace(*system['t_s'], len(steps) + 1)[1:]
ST = np.array([s['ST'] for s in steps])
plt.figure(figsize=(7, 3.5))
plt.plot(t, ST)
plt.legend(NAMES, ncol=3, fontsize=8)
plt.xlabel('time [h]'); plt.ylabel('total index ST of the impurity y3'); plt.ylim(0, 1)
plt.title('306 K'); plt.tight_layout(); plt.show()

### Decision: what to estimate

Keep a parameter if it matters for at least one output at at least one temperature; fix the rest
at their current value (`models['mutation']`). Save the results to reuse them later.

In [ ]:
ST_max = pd.concat([final_indices(r).xs('ST', axis=1, level=1) for r in (cold, hot)], axis=1).max(axis=1)
models['mutation']['M'] = [bool(v > 0.05) for v in ST_max]
print(pd.DataFrame({'largest ST': ST_max.round(3).values, 'estimate': models['mutation']['M']}, index=NAMES))
save_to_jac({'296K': cold, '306K': hot}, purpose='sensa')

**Try:** raise `samp` and watch the indices settle (Sobol estimates are noisy for small *N*);
use `par_d=1.05`; change the operating point `tii_n`.